# LemGendary SOTA Usage: VlmBlip2
Implementation guide for production-grade model integration.


## 1. PyTorch Standalone (FP32)
Best for local research, further training, or high-fidelity Python backends. This format includes the full architecture definition.


In [ ]:
import torch
from PIL import Image
import numpy as np

# 1. Load Standalone SOTA Model (Architecture + Weights)
# Precision: FP32 | Deployment: Local/Research
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = 'LemGendaryBLIP-2-Master.pt'
model = torch.load(model_path, map_location=device)
model.eval()

# 2. Prepare Input
img = Image.open('photo.jpg').convert('RGB').resize((224, 224))
input_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().unsqueeze(0).to(device) / 255.0

# 3. Normalization: ImageNet Stats (Standard for LemGendary Suite)
mean = torch.tensor([0.485, 0.456, 0.406]).to(device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).to(device).view(1, 3, 1, 1)
input_tensor = (input_tensor - mean) / std

with torch.no_grad():
    output = model(input_tensor)

print(f'Prediction Raw: {output.cpu().numpy()}')


## 2. ONNX Matrix (FP32 + External Weights)
Optimized for desktop deployment where precision is critical. Uses a decoupled `.data` file for stability.


In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image

# 1. Initialize High-Precision Session
# NOTE: Requires 'LemGendaryBLIP-2-Master_FP32.onnx.data' in the same folder!
# Precision: FP32 | Deployment: CPU/High-Accuracy Desktop
onnx_path = 'LemGendaryBLIP-2-Master_FP32.onnx'
session = ort.InferenceSession(onnx_path)

# 2. Prepare Input
img = Image.open('photo.jpg').convert('RGB').resize((224, 224))
input_data = (np.array(img).astype(np.float32) / 255.0 - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
input_data = input_data.transpose(2, 0, 1)[np.newaxis, :]

# 3. Inference
output = session.run(None, {'input': input_data})[0]
print(f'Prediction Raw: {output}')


## 3. ONNX Production (FP16 Embedded)
Production-ready standalone matrix. Optimized for WebGPU, mobile, and low-latency edge inference.


In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image

# 1. Initialize Production Session (Embedded Weights)
# Precision: FP16 | Deployment: WebGPU / Mobile / Edge / Production
onnx_path = 'LemGendaryBLIP-2-Master.onnx'
session = ort.InferenceSession(onnx_path)

# 2. Prepare Input
img = Image.open('photo.jpg').convert('RGB').resize((224, 224))
input_data = (np.array(img).astype(np.float32) / 255.0 - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
input_data = input_data.transpose(2, 0, 1)[np.newaxis, :]

# 3. Inference
output = session.run(None, {'input': input_data})[0]
print(f'Prediction Raw: {output}')
